In [ ]:
# =====================================================================
# Défi quotidien : Construire votre premier réseau neuronal sur MNIST
# Formation intensive en IA - 2026
# =====================================================================

import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist

# =====================================================================
# 1. Charger et prétraiter l'ensemble de données MNIST
# =====================================================================

print("--- Étape 1 : Chargement et prétraitement des données ---")

# Chargement des données brutes (entraînement et test)
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = mnist.load_data()

# Normalisation des valeurs de pixels (de [0, 255] à [0.0, 1.0])
x_train = x_train_raw.astype('float32') / 255.0
x_test = x_test_raw.astype('float32') / 255.0

# Encodage One-Hot pour les étiquettes (ex: 3 devient [0, 0, 0, 1, 0, 0, 0, 0, 0, 0])
y_train = to_categorical(y_train_raw, num_classes=10)
y_test = to_categorical(y_test_raw, num_classes=10)

print(f"Forme des données d'entraînement : {x_train.shape}")
print(f"Forme des données de test : {x_test.shape}")

# Visualisation de quelques exemples d'images
plt.figure(figsize=(10, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_train_raw[i], cmap='gray')
    plt.title(f"Label: {y_train_raw[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()


# =====================================================================
# 2. Construire un réseau neuronal entièrement connecté
# =====================================================================

print("\n--- Étape 2 : Construction du réseau de neurones ---")

# Définition de l'architecture séquentielle du modèle
model = models.Sequential([
    # Aplatir l'image d'entrée 28x28 en un vecteur 1D de 784 éléments
    layers.Flatten(input_shape=(28, 28), name="Couche_Entree"),
    
    # Première couche cachée avec 128 neurones et activation ReLU
    layers.Dense(128, activation='relu', name="Couche_Cachee_1"),
    
    # Deuxième couche cachée avec 64 neurones et activation ReLU (Ajout pour l'optimisation)
    layers.Dense(64, activation='relu', name="Couche_Cachee_2"),
    
    # Couche de sortie avec 10 neurones (un par chiffre) et activation Softmax pour la classification
    layers.Dense(10, activation='softmax', name="Couche_Sortie")
])

# Affichage du résumé structurel du modèle
model.summary()

# Compilation du modèle avec les spécifications requises
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# =====================================================================
# 3. Entraîner le réseau neuronal
# =====================================================================

print("\n--- Étape 3 : Entraînement du modèle ---")

# Entraînement sur 10 époques avec un échantillon de validation de 10%
history = model.fit(
    x_train, 
    y_train, 
    epochs=10, 
    batch_size=64, 
    validation_split=0.1,  # Permet de suivre les performances sur des données non vues
    verbose=1
)

# Visualisation des courbes d'apprentissage (Loss et Accuracy)
plt.figure(figsize=(12, 4))

# Graphique de la Précision (Accuracy)
plt.subplot(1, 2, i := 1)
plt.plot(history.history['accuracy'], label='Précision Entraînement', color='blue')
plt.plot(history.history['val_accuracy'], label='Précision Validation', color='orange')
plt.title("Évolution de la Précision au fil des Époques")
plt.xlabel("Époques")
plt.ylabel("Précision")
plt.legend()
plt.grid(True)

# Graphique de la Perte (Loss)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Perte Entraînement', color='blue')
plt.plot(history.history['val_loss'], label='Perte Validation', color='orange')
plt.title("Évolution de la Perte au fil des Époques")
plt.xlabel("Époques")
plt.ylabel("Perte (Loss)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


# =====================================================================
# 4. Évaluer les performances du modèle & Matrice de confusion
# =====================================================================

print("\n--- Étape 4 : Évaluation finale sur l'ensemble de test ---")

# Évaluation globale sur les données de test
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Perte sur le jeu de test : {test_loss:.4f}")
print(f"Précision sur le jeu de test : {test_acc * 100:.2f}%")

# Génération des prédictions du modèle sur le jeu de test
y_pred_probabilities = model.predict(x_test)
y_pred_labels = np.argmax(y_pred_probabilities, axis=1)

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test_raw, y_pred_labels)

# Affichage thermique de la matrice de confusion
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=list(range(10)), yticklabels=list(range(10)))
plt.title("Matrice de Confusion - Reconnaissance des Chiffres MNIST")
plt.xlabel("Chiffres Prédits par le Modèle")
plt.ylabel("Chiffres Réels")
plt.show()

# Analyse automatique des erreurs
print("\n--- Analyse des performances par chiffre ---")
diagonal = np.diag(cm)
total_per_digit = np.sum(cm, axis=1)
accuracy_per_digit = diagonal / total_per_digit

for digit in range(10):
    print(f"Précision pour le chiffre {digit} : {accuracy_per_digit[digit]*100:.2f}%")

hardest_digit = np.argmin(accuracy_per_digit)
print(f"\nLe chiffre avec lequel le modèle a le plus de difficultés est le : {hardest_digit} "
      f"({accuracy_per_digit[hardest_digit]*100:.2f}% de réussite)")

ModuleNotFoundError: No module named 'seaborn'